# Activity 1: RAGAS Evaluation with LangSmith Cost Analysis

This notebook builds two RAG pipelines over the same cat-health-guide PDF:
1. **Fireworks AI** — `gpt-oss-20b` chat + `qwen3-embedding-8b` embeddings
2. **OpenAI** — `gpt-4.1-mini` chat + `text-embedding-3-small` embeddings

Both pipelines are evaluated with **RAGAS** (faithfulness, answer relevancy, context precision, context recall) and instrumented with **LangSmith** for token usage and cost tracking.

## 1. Setup & Environment

Load environment variables and enable LangSmith tracing. Make sure your `.env` file has `FIREWORKS_API_KEY`, `OPENAI_API_KEY`, and `LANGSMITH_API_KEY` set.

In [1]:
import os
import getpass
from dotenv import load_dotenv

load_dotenv()

# Ensure all required API keys are present
if not os.environ.get("FIREWORKS_API_KEY"):
    os.environ["FIREWORKS_API_KEY"] = getpass.getpass("Enter your Fireworks API key: ")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API key: ")
if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass.getpass("Enter your LangSmith API key: ")

# Enable LangSmith tracing
os.environ["LANGSMITH_TRACING"] = "true"
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = os.environ["LANGSMITH_API_KEY"]
os.environ.setdefault("LANGSMITH_PROJECT", "ragas-evaluation")
os.environ.setdefault("LANGCHAIN_PROJECT", os.environ["LANGSMITH_PROJECT"])

print("Environment configured. LangSmith tracing enabled.")
print(f"Project: {os.environ.get('LANGSMITH_PROJECT')}")

Environment configured. LangSmith tracing enabled.
Project: AIE Course


## 2. Build Shared Document Store

Load the cat-health-guide PDF, split into chunks, and build **two** in-memory Qdrant vector stores — one with Fireworks embeddings, one with OpenAI embeddings. Both use the same chunks so retrieval quality differences come from the embedding model, not the content.

In [2]:
import tiktoken
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai.embeddings import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore

FIREWORKS_BASE_URL = "https://api.fireworks.ai/inference/v1"

# --- Load and split PDF ---
loader = PyMuPDFLoader("data/cat-health-guide.pdf")
documents = loader.load()
print(f"Loaded {len(documents)} pages from cat-health-guide.pdf")

def tiktoken_len(text: str) -> int:
    tokens = tiktoken.encoding_for_model("gpt-4o").encode(text)
    return len(tokens)

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=750, chunk_overlap=0, length_function=tiktoken_len
)
chunks = text_splitter.split_documents(documents)
print(f"Split into {len(chunks)} chunks")

# --- Fireworks embeddings vector store ---
fireworks_embedding = OpenAIEmbeddings(
    model=os.environ.get("FIREWORKS_EMBEDDING_MODEL", "accounts/fireworks/models/qwen3-embedding-8b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
    check_embedding_ctx_length=False,
    dimensions=4096,
)
fireworks_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=fireworks_embedding,
    location=":memory:",
    collection_name="fireworks_rag",
)
fireworks_retriever = fireworks_vectorstore.as_retriever()
print("Fireworks vector store built.")

# --- OpenAI embeddings vector store ---
openai_embedding = OpenAIEmbeddings(
    model="text-embedding-3-small",
    openai_api_key=os.environ["OPENAI_API_KEY"],
)
openai_vectorstore = QdrantVectorStore.from_documents(
    documents=chunks,
    embedding=openai_embedding,
    location=":memory:",
    collection_name="openai_rag",
)
openai_retriever = openai_vectorstore.as_retriever()
print("OpenAI vector store built.")

Loaded 22 pages from cat-health-guide.pdf
Split into 42 chunks
Fireworks vector store built.
OpenAI vector store built.


## 3. Build Two RAG Pipelines

Each pipeline: retrieve relevant chunks → generate answer constrained to context. Both use the same prompt template. We wrap each in a function that returns the answer and the retrieved context documents.

In [3]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

RAG_PROMPT_TEMPLATE = (
    "\n#CONTEXT:\n{context}\n\nQUERY:\n{query}\n\n"
    "Use the provided context to answer the provided user query. "
    "Only use the provided context to answer the query. If you do not know the answer, "
    "or it's not contained in the provided context respond with \"I don't know\""
)
chat_prompt = ChatPromptTemplate.from_messages([("human", RAG_PROMPT_TEMPLATE)])

# --- Fireworks RAG pipeline ---
fireworks_llm = ChatOpenAI(
    model=os.environ.get("FIREWORKS_CHAT_MODEL", "accounts/fireworks/models/gpt-oss-20b"),
    openai_api_key=os.environ["FIREWORKS_API_KEY"],
    openai_api_base=FIREWORKS_BASE_URL,
)

def fireworks_rag(query: str) -> dict:
    """Run the Fireworks RAG pipeline. Returns answer + retrieved contexts."""
    docs = fireworks_retriever.invoke(query)
    context_text = "\n\n".join(doc.page_content for doc in docs)
    chain = chat_prompt | fireworks_llm | StrOutputParser()
    answer = chain.invoke({"query": query, "context": context_text})
    return {"answer": answer, "contexts": [doc.page_content for doc in docs]}

# --- OpenAI RAG pipeline ---
openai_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    openai_api_key=os.environ["OPENAI_API_KEY"],
)

def openai_rag(query: str) -> dict:
    """Run the OpenAI RAG pipeline. Returns answer + retrieved contexts."""
    docs = openai_retriever.invoke(query)
    context_text = "\n\n".join(doc.page_content for doc in docs)
    chain = chat_prompt | openai_llm | StrOutputParser()
    answer = chain.invoke({"query": query, "context": context_text})
    return {"answer": answer, "contexts": [doc.page_content for doc in docs]}

# Quick smoke test
test_q = "What vaccinations are recommended for kittens?"
fw_test = fireworks_rag(test_q)
print(f"Fireworks answer: {fw_test['answer'][:150]}...")
oai_test = openai_rag(test_q)
print(f"OpenAI answer: {oai_test['answer'][:150]}...")

Fireworks answer: **Vaccinations recommended for kittens**

*Core vaccines* (recommended for all kittens):  
- Rabies virus  
- Feline herpesvirus type 1 (FHV‑1)  
- Fe...
OpenAI answer: The recommended vaccinations for kittens include core vaccines against rabies virus, feline herpesvirus type 1 (FHV-1), feline calicivirus (FCV), and ...


## 4. Generate Evaluation Dataset

Use OpenAI `gpt-4.1-mini` to auto-generate Q&A pairs from the cat health PDF chunks. Each item has a `question`, `ground_truth` answer, and the source `contexts`.

In [4]:
import json
import re
import random
from langchain_core.messages import HumanMessage, SystemMessage

gen_llm = ChatOpenAI(
    model="gpt-4.1-mini",
    openai_api_key=os.environ["OPENAI_API_KEY"],
    temperature=0.7,
)

gen_prompt = """You are a veterinary expert creating a test dataset for a RAG system about cat health.
Given the following text chunk from a cat health guide, generate 2 question-answer pairs.
Questions should be answerable using ONLY the information in the text.
Answers should be concise and factual.

Text chunk:
{chunk}

Respond as a JSON list of objects with keys "question" and "answer". Do not include any other text or markdown formatting."""

def strip_json_fences(text: str) -> str:
    """Remove markdown ```json fences if present."""
    text = text.strip()
    if text.startswith("```"):
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
    return text.strip()

# Sample chunks to generate questions from
sampled_chunks = random.sample(chunks, min(8, len(chunks)))

eval_data = []
for chunk in sampled_chunks:
    msg = HumanMessage(content=gen_prompt.format(chunk=chunk.page_content[:1500]))
    resp = gen_llm.invoke([msg])
    try:
        cleaned = strip_json_fences(resp.content)
        pairs = json.loads(cleaned)
        for pair in pairs:
            eval_data.append({
                "question": pair["question"],
                "ground_truth": pair["answer"],
                "source_context": chunk.page_content,
            })
    except (json.JSONDecodeError, KeyError):
        print(f"Skipped a chunk due to parse error: {resp.content[:100]}...")

print(f"Generated {len(eval_data)} Q&A pairs")
for i, item in enumerate(eval_data[:3]):
    print(f"\n--- Q{i+1}: {item['question']}")
    print(f"    A: {item['ground_truth'][:100]}...")

Generated 16 Q&A pairs

--- Q1: What impact do nutritional interventions have on serum symmetric dimethylarginine and creatinine concentrations in geriatric cats?
    A: Nutritional interventions have a positive impact on serum symmetric dimethylarginine and creatinine ...

--- Q2: Where can one find the 2019 AAHA dental care guidelines for dogs and cats?
    A: The 2019 AAHA dental care guidelines for dogs and cats are available at https://www.aaha.org/globala...

--- Q3: Where can information about companion animal dental scaling without anesthesia be found?
    A: At the American Veterinary Dental College website: https://avdc.org/?s¼Dental1scaling1without1anesth...


## 5. Run RAGAS Evaluation on Both Pipelines

Run each pipeline on all evaluation questions, then score with RAGAS using `gpt-4.1-mini` as the evaluator LLM for consistent judging.

In [5]:
import asyncio
import pandas as pd
from concurrent.futures import ThreadPoolExecutor
from openai import OpenAI
import instructor
from ragas.llms import llm_factory
from ragas.metrics.collections import (
    Faithfulness,
    AnswerAccuracy,
    ContextRecall,
    ContextPrecision,
)

# Build RAGAS judge LLM using the factory API with a synchronous client
judge_llm = llm_factory(
    "gpt-4.1-mini",
    provider="openai",
    client=OpenAI(api_key=os.environ["OPENAI_API_KEY"]),
    mode=instructor.Mode.TOOLS,
    max_tokens=2048,
)
judge_llm.model_args = {"max_tokens": 2048, "max_retries": 3}

# The metrics call agenerate() but our client is synchronous.
# Bridge the async boundary by delegating to the sync generate() via to_thread.
async def _agenerate_adapter(prompt, response_model):
    return await asyncio.to_thread(judge_llm.generate, prompt=prompt, response_model=response_model)

judge_llm.agenerate = _agenerate_adapter

# Instantiate metrics with the judge LLM
rag_metrics = {
    "faithfulness": Faithfulness(llm=judge_llm),
    "answer_accuracy": AnswerAccuracy(llm=judge_llm),
    "context_recall": ContextRecall(llm=judge_llm),
    "context_precision": ContextPrecision(llm=judge_llm),
}

def run_ragas_async(async_function, *args, **kwargs):
    with ThreadPoolExecutor(max_workers=1) as executor:
        return executor.submit(asyncio.run, async_function(*args, **kwargs)).result()

def run_rag_pipeline(rag_fn, eval_items, provider_name):
    """Run a RAG pipeline on eval questions and collect results."""
    rows = []
    for item in eval_items:
        result = rag_fn(item["question"])
        rows.append({
            "provider": provider_name,
            "question": item["question"],
            "answer": result["answer"],
            "contexts": result["contexts"],
            "ground_truth": item["ground_truth"],
        })
    return rows

async def score_rows(rows):
    """Score each row with RAGAS metrics."""
    score_rows = []
    for i, row in enumerate(rows, start=1):
        scores = {}
        scores["faithfulness"] = (await rag_metrics["faithfulness"].ascore(
            user_input=row["question"],
            response=row["answer"],
            retrieved_contexts=row["contexts"],
        )).value
        scores["answer_accuracy"] = (await rag_metrics["answer_accuracy"].ascore(
            user_input=row["question"],
            response=row["answer"],
            reference=row["ground_truth"],
        )).value
        scores["context_recall"] = (await rag_metrics["context_recall"].ascore(
            user_input=row["question"],
            retrieved_contexts=row["contexts"],
            reference=row["ground_truth"],
        )).value
        scores["context_precision"] = (await rag_metrics["context_precision"].ascore(
            user_input=row["question"],
            retrieved_contexts=row["contexts"],
            reference=row["ground_truth"],
        )).value
        scores["case"] = i
        scores["provider"] = row["provider"]
        scores["question"] = row["question"][:60]
        score_rows.append(scores)
        print(f"  [{row['provider']}] Q{i} scored")
    return pd.DataFrame(score_rows)

print("Running Fireworks pipeline on evaluation questions...")
fw_rows = run_rag_pipeline(fireworks_rag, eval_data, "Fireworks (gpt-oss-20b)")

print("Running OpenAI pipeline on evaluation questions...")
oai_rows = run_rag_pipeline(openai_rag, eval_data, "OpenAI (gpt-4.1-mini)")

print("\nScoring Fireworks results with RAGAS...")
fw_scores = run_ragas_async(score_rows, fw_rows)

print("Scoring OpenAI results with RAGAS...")
oai_scores = run_ragas_async(score_rows, oai_rows)

combined = pd.concat([fw_scores, oai_scores], ignore_index=True)
metric_cols = ["faithfulness", "answer_accuracy", "context_recall", "context_precision"]
display(combined[["provider", "question"] + metric_cols].round(4))

print("\n=== Average Scores by Provider ===")
avg = combined.groupby("provider")[metric_cols].mean().round(4)
display(avg)

print("\n=== RAGAS Evaluation Complete ===")

Running Fireworks pipeline on evaluation questions...
Running OpenAI pipeline on evaluation questions...

Scoring Fireworks results with RAGAS...
  [Fireworks (gpt-oss-20b)] Q1 scored
  [Fireworks (gpt-oss-20b)] Q2 scored
  [Fireworks (gpt-oss-20b)] Q3 scored
  [Fireworks (gpt-oss-20b)] Q4 scored
  [Fireworks (gpt-oss-20b)] Q5 scored
  [Fireworks (gpt-oss-20b)] Q6 scored
  [Fireworks (gpt-oss-20b)] Q7 scored
  [Fireworks (gpt-oss-20b)] Q8 scored
  [Fireworks (gpt-oss-20b)] Q9 scored
  [Fireworks (gpt-oss-20b)] Q10 scored
  [Fireworks (gpt-oss-20b)] Q11 scored
  [Fireworks (gpt-oss-20b)] Q12 scored
  [Fireworks (gpt-oss-20b)] Q13 scored
  [Fireworks (gpt-oss-20b)] Q14 scored
  [Fireworks (gpt-oss-20b)] Q15 scored
  [Fireworks (gpt-oss-20b)] Q16 scored
Scoring OpenAI results with RAGAS...
  [OpenAI (gpt-4.1-mini)] Q1 scored
  [OpenAI (gpt-4.1-mini)] Q2 scored
  [OpenAI (gpt-4.1-mini)] Q3 scored
  [OpenAI (gpt-4.1-mini)] Q4 scored
  [OpenAI (gpt-4.1-mini)] Q5 scored
  [OpenAI (gpt-4.1-min

,provider,question,faithfulness,answer_accuracy,context_recall,context_precision
0,Fireworks (gpt-oss-20b),What impact do nutritional interventions have ...,1.0000,0.75,1.0,0.7500
1,Fireworks (gpt-oss-20b),Where can one find the 2019 AAHA dental care g...,1.0000,0.00,0.0,0.0000
2,Fireworks (gpt-oss-20b),Where can information about companion animal d...,1.0000,0.50,1.0,1.0000
3,Fireworks (gpt-oss-20b),What organization provides guidelines on paras...,1.0000,1.00,1.0,1.0000
4,Fireworks (gpt-oss-20b),How is the resting energy requirement (RER) fo...,0.5000,0.00,1.0,1.0000
5,Fireworks (gpt-oss-20b),Why are prescription diets recommended for ove...,0.6667,1.00,1.0,0.8333
6,Fireworks (gpt-oss-20b),What environmental management approach can hel...,0.7143,0.25,1.0,1.0000
7,Fireworks (gpt-oss-20b),What considerations should be made for senior ...,0.7500,0.50,1.0,1.0000
8,Fireworks (gpt-oss-20b),Which guideline provides recommendations for t...,1.0000,1.00,1.0,1.0000
9,Fireworks (gpt-oss-20b),What is the purpose of measuring plasma N-term...,1.0000,1.00,1.0,1.0000



=== Average Scores by Provider ===


,faithfulness,answer_accuracy,context_recall,context_precision
provider,,,,
Fireworks (gpt-oss-20b),0.8811,0.6875,0.9375,0.8212
OpenAI (gpt-4.1-mini),0.9688,0.7188,1.0000,0.8472



=== RAGAS Evaluation Complete ===


## 6. LangSmith Cost & Token Analysis

Query the LangSmith API for traces from this session to extract token usage and cost per pipeline run.

In [12]:
from langsmith import Client
from datetime import datetime, timedelta, timezone

langsmith_client = Client(api_key=os.environ["LANGSMITH_API_KEY"])
project_name = os.environ.get("LANGSMITH_PROJECT", "ragas-evaluation")

# Fetch recent runs from the project
runs = list(langsmith_client.list_runs(
    project_name=project_name,
    start_time=datetime.now(timezone.utc) - timedelta(hours=2),
    is_root=True,
))
print(f"Retrieved {len(runs)} root runs from LangSmith project '{project_name}'")

# Extract token usage and cost from each run
usage_records = []
for run in runs:
    # Walk the run tree to aggregate token usage across all child runs
    total_prompt = 0
    total_completion = 0
    total_tokens = 0
    total_cost = 0.0

    all_runs = [run]
    # Get child runs
    try:
        child_runs = list(langsmith_client.list_runs(
            project_name=project_name,
            trace_id=run.trace_id,
        ))
        all_runs = child_runs
    except Exception:
        pass

    for r in all_runs:
        total_prompt += int(r.prompt_tokens or 0)
        total_completion += int(r.completion_tokens or 0)
        total_tokens += int(r.total_tokens or 0)
        total_cost += float(r.total_cost or 0.0)

    # Identify provider from the run name or model — check all runs in the trace
    run_name = run.name or ""
    model_info = ""
    for r in all_runs:
        if r.extra and "metadata" in r.extra:
            mi = r.extra["metadata"].get("ls_model_name", "")
            if mi:
                model_info = mi
                break

    provider = "unknown"
    if "gpt-oss" in str(model_info).lower() or "fireworks" in str(model_info).lower():
        provider = "Fireworks"
    elif "gpt-4.1-mini" in str(model_info) or "gpt-4" in str(model_info).lower():
        provider = "OpenAI"

    usage_records.append({
        "run_name": run_name[:50],
        "provider": provider,
        "prompt_tokens": total_prompt,
        "completion_tokens": total_completion,
        "total_tokens": total_tokens,
        "cost_usd": round(total_cost, 6),
    })

usage_df = pd.DataFrame(usage_records)
if not usage_df.empty:
    display(usage_df)

    print("\n=== Token & Cost Summary by Provider ===")
    summary = usage_df.groupby("provider").agg({
        "prompt_tokens": "sum",
        "completion_tokens": "sum",
        "total_tokens": "sum",
        "cost_usd": "sum",
    }).round(4)
    display(summary)
else:
    print("No runs with usage data found. Make sure LangSmith tracing is enabled and runs have completed.")

Retrieved 152 root runs from LangSmith project 'AIE Course'


,run_name,provider,prompt_tokens,completion_tokens,total_tokens,cost_usd
0,RunnableSequence,OpenAI,4602,442,5044,0.002548
1,VectorStoreRetriever,unknown,0,0,0,0.000000
2,RunnableSequence,OpenAI,5980,68,6048,0.002501
3,VectorStoreRetriever,unknown,0,0,0,0.000000
4,RunnableSequence,OpenAI,6000,226,6226,0.002762
...,...,...,...,...,...,...
147,ChatOpenAI,OpenAI,435,121,556,0.000368
148,RunnableSequence,OpenAI,5302,302,5604,0.002604
149,VectorStoreRetriever,unknown,0,0,0,0.000000
150,RunnableSequence,Fireworks,5776,1318,7094,0.000000



=== Token & Cost Summary by Provider ===


,prompt_tokens,completion_tokens,total_tokens,cost_usd
provider,,,,
Fireworks,162538,21372,183910,0.0000
OpenAI,186396,6078,192474,0.0831
unknown,0,0,0,0.0000
